# ASP: verify the seal and replicate in the browser

This notebook checks the repository [ulissesflores/anticipating-shadow-points](https://github.com/ulissesflores/anticipating-shadow-points) without installing anything locally. It calls no model, and a run takes about two minutes. It works in four steps:

1. Clone the repository at the reference set below.
2. **Verify the committed seal.** It recomputes the SHA-256 of every sealed file (code, tests, frozen inputs, data and derived results) and checks the single `chain_hash` against `output/provenance.json`.
3. **Replicate.** `run_all.py` derives every published number again from the committed data, runs the test suite (each published number is an assertion), seals the result and verifies it.
4. Confirm that the replication changed nothing sealed, and print the headline numbers.

If any file was altered after sealing, step 2 or step 4 fails.

In [ ]:
REF = "main"  # a tag such as "v2.0.0" pins an exact release
!git clone --quiet --depth 1 --branch {REF} https://github.com/ulissesflores/anticipating-shadow-points.git asp
%cd asp
!git log -1 --format="commit %H  (%cd)"

In [ ]:
%pip install --quiet -r requirements.txt

## 1. Verify the committed seal

In [ ]:
!python make_provenance.py --verify

## 2. Replicate: re-derive, test, re-seal, verify

In [ ]:
!python run_all.py

## 3. Nothing sealed changed

The figures and the lockfile are informational and are not hashed, because they vary by machine. Every other file must match the commit byte for byte.

In [ ]:
import json, subprocess
changed = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.split("\n")
sealed_changes = [l for l in changed if l.strip() and "output/figures/" not in l and "requirements.lock" not in l]
print("sealed files changed by the replication:", sealed_changes or "none")
assert not sealed_changes, "the replication does not match the committed results"
prov = json.load(open("output/provenance.json"))
print("chain_hash:", prov.get("chain_hash"))

## 4. Headline numbers

In [ ]:
results = json.load(open("output/results.json"))
print(json.dumps(results, indent=1)[:3000])

## 5. The 2026-11 study ledger (when present)

Each artifact of the pre-registered study is chained to the previous one by hash. Verification recomputes every link and every file hash.

In [ ]:
import os
if os.path.exists("benchmark/study-2026-11/ledger.jsonl"):
    !python benchmark/study-2026-11/ledger.py verify
else:
    print("no study ledger at this reference")